# Train and Evaluate (Notebook Version)

This notebook combines the logic from `train.py` and `eval.py` into a single, reproducible workflow.

In [18]:
from pathlib import Path
from datetime import datetime
import sys
from types import SimpleNamespace
import numpy as np
import torch
from torch.utils.data import DataLoader
from sklearn.preprocessing import LabelEncoder
import pandas as pd
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
   
from utils.seed import set_seed
from utils.device import get_device

from data.preprocessing import (
    FEATURE_COLS,
    get_feature_columns,
    create_windows,
    clean_features,
    normalize_features,
)
from data.motion_dataset import MotionDataset
from model.bilstm import BiLSTM
from training.loss import build_loss
from training.trainer import train_epoch
from callbacks.early_stopping import EarlyStopping
from callbacks.lr_scheduler import ReduceLROnPlateau

# -------------------------
# Config (edit as needed)
# -------------------------
cfg = {
    "seed": 42,
    "device": "auto",
    "data": {
        "sensor_type": "Segment Velocity",
        "video_suffix": None,
        "window_size": 30,
        "batch_size": 32,
        "num_workers": 3,
    },
    "model": {
        "hidden_size": 128,
        "num_layers": 2,
    },
    "train": {
        "epochs": 2,
        "lr": 1e-4,
        "grad_clip": 1.0,
        "weight_decay": 1e-5,
    },
    "loss": {
        "type": "auto",  # auto | bce | ce
    },
    "callbacks": {
        "early_stopping": {
            "enabled": True,
            "monitor": "acc",
            "patience": 10,
            "min_delta": 0.001,
        },
        "reduce_lr": {
            "enabled": True,
            "monitor": "loss",
            "factor": 0.5,
            "patience": 5,
            "min_lr": 1e-6,
        },
    },
}


NOTEBOOK_DIR = Path.cwd()
DATA_DIR = PROJECT_ROOT / "output_data"
run_dir = NOTEBOOK_DIR / "results" / datetime.now().strftime("%Y-%m-%d/%H-%M-%S")
run_dir.mkdir(parents=True, exist_ok=True)

set_seed(cfg["seed"])
device = get_device(cfg["device"])

print("Results dir:", run_dir)

Results dir: d:\Swin documents\COS40007-Project\backend\notebook\results\2026-02-03\11-46-10


# Preprocess

In [19]:
# -------------------------
# Load training data (P1)
# -------------------------
df_boning = pd.read_csv(DATA_DIR / "P1_boning.csv")
df_slicing = pd.read_csv(DATA_DIR / "P1_slicing.csv")

df = pd.concat([df_boning, df_slicing], ignore_index=True)

# filter sensor
df = df[df["sensor_type"] == cfg["data"]["sensor_type"]]

# optional: filter video suffix (e.g. 001)
if cfg["data"]["video_suffix"] is not None:
    df = df[df["video_id"].str.endswith(cfg["data"]["video_suffix"])]

assert df["activity_type"].nunique() > 1, "Only one class left after filtering"

print("Activity counts:")
print(df["activity_type"].value_counts())
print("Total columns:", len(df.columns))
print("Unique columns:", len(set(df.columns)))
extra_cols = [
    c for c in df.columns
    if c not in FEATURE_COLS
    and c not in [
        "Frame",
        "Label",
        "person_id",
        "activity_type",
        "knife_sharpness_score",
        "sensor_type",
        "video_id",
    ]
]
missing_cols = [c for c in FEATURE_COLS if c not in df.columns]
print("Extra non-feature columns:", extra_cols)
print("Missing feature columns:", missing_cols)

feature_cols = get_feature_columns(df)

X, y = create_windows(df, feature_cols, cfg["data"]["window_size"])
X = clean_features(X)
X, scaler = normalize_features(X)

# Feature stats after normalization
X_flat = X.reshape(-1, X.shape[-1])
stats = pd.DataFrame({
    "min": X_flat.min(axis=0),
    "max": X_flat.max(axis=0),
    "mean": X_flat.mean(axis=0),
    "std": X_flat.std(axis=0),
}, index=feature_cols)
print("Normalized feature stats:")
print(stats)

encoder = LabelEncoder()
y = encoder.fit_transform(y)

print("Train samples:", len(X))

Activity counts:
activity_type
boning     141397
slicing    125527
Name: count, dtype: int64
Total columns: 76
Unique columns: 76
Extra non-feature columns: []
Missing feature columns: []
Normalized feature stats:
                   min        max          mean  std
Pelvis x      0.000000   0.000000  0.000000e+00  0.0
Pelvis y      0.000000   0.000000  0.000000e+00  0.0
Pelvis z      0.000000   0.000000  0.000000e+00  0.0
L5 x         -8.282658   8.275899 -7.787253e-14  1.0
L5 y         -6.972686   6.257574  9.347622e-14  1.0
...                ...        ...           ...  ...
Left Foot y -16.245943  12.723246  4.257774e-16  1.0
Left Foot z -19.572026  22.913207  2.023509e-16  1.0
Left Toe x  -12.830186  10.338516  3.605310e-16  1.0
Left Toe y  -16.245943  12.723246  4.257774e-16  1.0
Left Toe z  -19.572026  22.913207  2.023509e-16  1.0

[69 rows x 4 columns]
Train samples: 266518


# Train

In [20]:
# -------------------------
# Dataset / Loader
# -------------------------
print("Feature columns used:", feature_cols)
print("X shape:", X.shape, "y shape:", y.shape)
print("Sample window shape:", X[0].shape)
print("Sample window (first 2 timesteps):")
print(X[0][:2])

dataset = MotionDataset(X, y)
loader = DataLoader(
    dataset,
    batch_size=cfg["data"]["batch_size"],
    shuffle=True,
    num_workers=cfg["data"]["num_workers"],
    pin_memory=True,
)

# -------------------------
# Model / Optim / Loss
# -------------------------
model = BiLSTM(
    input_size=X.shape[2],
    hidden_size=cfg["model"]["hidden_size"],
    num_classes=len(encoder.classes_),
    num_layers=cfg["model"]["num_layers"],
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Model parameters:", total_params, "(trainable:", trainable_params, ")")

try:
    from torchinfo import summary
    summary(model, input_size=(1, X.shape[1], X.shape[2]))
except Exception:
    print("Model architecture:")
    print(model)

sample_input = torch.tensor(X[:1], dtype=torch.float32).to(device)
sample_output = model(sample_input)
print("Sample model input shape:", sample_input.shape)
print("Sample model output shape:", sample_output.shape)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=cfg["train"]["lr"],
    weight_decay=cfg["train"]["weight_decay"],
)

loss_cfg = SimpleNamespace(**cfg["loss"])
criterion, mode = build_loss(loss_cfg, len(encoder.classes_))

# -------------------------
# Callbacks
# -------------------------
es = None
if cfg["callbacks"]["early_stopping"]["enabled"]:
    es = EarlyStopping(
        monitor=cfg["callbacks"]["early_stopping"]["monitor"],
        patience=cfg["callbacks"]["early_stopping"]["patience"],
        min_delta=cfg["callbacks"]["early_stopping"]["min_delta"],
    )

rlr = None
if cfg["callbacks"]["reduce_lr"]["enabled"]:
    rlr = ReduceLROnPlateau(
        optimizer=optimizer,
        monitor=cfg["callbacks"]["reduce_lr"]["monitor"],
        factor=cfg["callbacks"]["reduce_lr"]["factor"],
        patience=cfg["callbacks"]["reduce_lr"]["patience"],
        min_lr=cfg["callbacks"]["reduce_lr"]["min_lr"],
    )

Feature columns used: ['Pelvis x', 'Pelvis y', 'Pelvis z', 'L5 x', 'L5 y', 'L5 z', 'L3 x', 'L3 y', 'L3 z', 'T12 x', 'T12 y', 'T12 z', 'T8 x', 'T8 y', 'T8 z', 'Neck x', 'Neck y', 'Neck z', 'Head x', 'Head y', 'Head z', 'Right Shoulder x', 'Right Shoulder y', 'Right Shoulder z', 'Right Upper Arm x', 'Right Upper Arm y', 'Right Upper Arm z', 'Right Forearm x', 'Right Forearm y', 'Right Forearm z', 'Right Hand x', 'Right Hand y', 'Right Hand z', 'Left Shoulder x', 'Left Shoulder y', 'Left Shoulder z', 'Left Upper Arm x', 'Left Upper Arm y', 'Left Upper Arm z', 'Left Forearm x', 'Left Forearm y', 'Left Forearm z', 'Left Hand x', 'Left Hand y', 'Left Hand z', 'Right Upper Leg x', 'Right Upper Leg y', 'Right Upper Leg z', 'Right Lower Leg x', 'Right Lower Leg y', 'Right Lower Leg z', 'Right Foot x', 'Right Foot y', 'Right Foot z', 'Right Toe x', 'Right Toe y', 'Right Toe z', 'Left Upper Leg x', 'Left Upper Leg y', 'Left Upper Leg z', 'Left Lower Leg x', 'Left Lower Leg y', 'Left Lower Leg z',

In [21]:
# -------------------------
# Training loop
# -------------------------
for epoch in range(cfg["train"]["epochs"]):
    loss, acc = train_epoch(
        model=model,
        loader=loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        grad_clip=cfg["train"]["grad_clip"],
        mode=mode,
    )

    print(f"[{epoch+1}/{cfg['train']['epochs']}] loss={loss:.4f} | acc={acc:.4f}")

    logs = {
        "loss": loss,
        "acc": acc,
        "model": model,
    }

    if es:
        es.on_epoch_end(epoch, logs)
        if es.stop:
            print("Early stopping")
            break

    if rlr:
        rlr.on_epoch_end(epoch, logs)

[1/2] loss=0.0694 | acc=0.9762
[2/2] loss=0.0138 | acc=0.9961


In [22]:
# -------------------------
# Save final artifacts
# -------------------------
save_path = run_dir / "last_model.pt"
torch.save(model.state_dict(), save_path)
print("Saved model to:", save_path)

Saved model to: d:\Swin documents\COS40007-Project\backend\notebook\results\2026-02-03\11-46-10\last_model.pt


# Evaluation

In [23]:
# -------------------------
# Evaluation (P2)
# -------------------------
df_boning = pd.read_csv(DATA_DIR / "P2_boning.csv")
df_slicing = pd.read_csv(DATA_DIR / "P2_slicing.csv")

df = pd.concat([df_boning, df_slicing], ignore_index=True)

# filter sensor
df = df[df["sensor_type"] == cfg["data"]["sensor_type"]]

# optional video suffix
if cfg["data"]["video_suffix"] is not None:
    df = df[df["video_id"].str.endswith(cfg["data"]["video_suffix"])]

assert df["activity_type"].nunique() > 1, "Only one class left after filtering"

feature_cols = get_feature_columns(df)
X, y = create_windows(df, feature_cols, cfg["data"]["window_size"])
X = clean_features(X)

# normalize using training scaler
X = scaler.transform(X.reshape(-1, X.shape[-1])).reshape(X.shape)
y = encoder.transform(y)

dataset = MotionDataset(X, y)
loader = DataLoader(
    dataset,
    batch_size=cfg["data"]["batch_size"],
    shuffle=False,
    num_workers=cfg["data"]["num_workers"],
)

model.eval()

criterion, mode = build_loss(loss_cfg, len(encoder.classes_))

total_loss = 0.0
correct = 0
total = 0
all_preds = []
all_targets = []

with torch.no_grad():
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        outputs = model(X_batch)

        if mode == "ce":
            loss = criterion(outputs, y_batch)
            preds = outputs.argmax(dim=1)
        else:
            loss = criterion(outputs.squeeze(), y_batch.float())
            preds = (outputs > 0.5).long()

        total_loss += loss.item() * y_batch.size(0)
        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)
        all_preds.append(preds.cpu())
        all_targets.append(y_batch.cpu())

avg_loss = total_loss / total
acc = correct / total

print("Evaluation results (P2)")
print(f"Loss: {avg_loss:.4f}")
print(f"Accuracy: {acc:.4f}")

y_true = torch.cat(all_targets).numpy()
y_pred = torch.cat(all_preds).numpy()
wrong_idx = np.where(y_true != y_pred)[0]
print("Wrong predictions:", len(wrong_idx))
show_n = min(10, len(wrong_idx))
for i in wrong_idx[:show_n]:
    true_label = encoder.inverse_transform([y_true[i]])[0]
    pred_label = encoder.inverse_transform([y_pred[i]])[0]
    print(f"idx={i} true={true_label} pred={pred_label}")

print("Evaluation complete")

Evaluation results (P2)
Loss: 2.4052
Accuracy: 23.9757


MemoryError: Unable to allocate 64.4 GiB for an array with shape (263021, 263021) and data type bool